In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import sqlite3
import numpy as np

# 1. Define the exact same architecture as your Flask app
# Your Flask app does: np.dot(weights, img_flat)
# This is equivalent to a Linear layer with NO bias.
class SimpleBrain(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(784, 26, bias=False)

    def forward(self, x):
        x = x.view(-1, 784) # Flatten the 28x28 image
        return self.fc(x)

def train_and_export():
    print("Downloading EMNIST dataset (this might take a minute the first time)...")
    
    # EMNIST images are rotated 90 degrees and flipped by default. 
    # This transform fixes them so they match what you draw on your HTML canvas.
    transform = transforms.Compose([
        transforms.ToTensor(),
        lambda x: x.transpose(1, 2) 
    ])

    # Load the "letters" split
    train_data = datasets.EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
    train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)

    model = SimpleBrain()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.005)

    epochs = 3
    print(f"Starting training for {epochs} epochs...")

    for epoch in range(epochs):
        total_loss = 0
        for images, labels in train_loader:
            # EMNIST 'letters' labels are 1-26 (1=A, 26=Z). 
            # We need them to be 0-25 to match Python indexing and your Flask app.
            labels = labels - 1 

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

    print("Training complete! Extracting weights...")

    # 2. Extract the trained weights back to a numpy array
    # Shape will be (26, 784)
    trained_weights = model.fc.weight.detach().numpy().astype(np.float32)

    # 3. Inject the smart weights into your SQLite Memory Block
    conn = sqlite3.connect('brain.db')
    c = conn.cursor()
    
    # Update the existing row where layer='fc_weight'
    c.execute("UPDATE model_store SET blob = ? WHERE layer = 'fc_weight'", 
              (trained_weights.tobytes(),))
    
    conn.commit()
    conn.close()
    print("✅ Smart weights successfully transplanted into brain.db!")

if __name__ == "__main__":
    train_and_export()

100.0%


Starting training for 3 epochs...
Epoch 1/3 | Loss: 1.1802
Epoch 2/3 | Loss: 1.1297
Epoch 3/3 | Loss: 1.1215
Training complete! Extracting weights...
✅ Smart weights successfully transplanted into brain.db!
